# 4.1 · 用 Python SDK 写第一个服务器

**Your first server with the Python SDK**

> **学完本课你能：** 用 Python SDK 构建、运行并连接一个真实的 MCP 服务器——由类型注解生成工具、在生命周期钩子中加载状态、通过 Context 报告进度、支持 stdio 与 Streamable HTTP，并写出一份可用的 Claude Desktop 配置。

官方阅读材料：[Build an MCP server](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-server) · [SDKs](https://modelcontextprotocol.io/docs/2026-07-28/sdk)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 一个假的美国国家气象局（无需联网）
官方教程会调用 `api.weather.gov`。为了保持离线和结果确定，我们把一个很小的、结构与 NWS 一致的数据集保存到临时文件夹中。服务器通过生命周期钩子（Lifespan）<strong>只加载一次</strong>这份数据。它的路径以<strong>绝对</strong>路径的形式通过环境变量传入，和你在 `claude_desktop_config.json` 中的做法完全一样。

In [ ]:
import json
import os
import sys
import tempfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)
workdir = Path(tempfile.mkdtemp(prefix="weather-"))
server_path = workdir / "weather.py"
data_path = workdir / "nws_sample.json"

nws = {
    "alerts": {
        "CA": [{"event": "Excessive Heat Warning", "areaDesc": "Sacramento Valley", "severity": "Severe",
                "description": "Dangerously hot conditions with temperatures up to 110.",
                "instruction": "Drink plenty of fluids and stay in an air-conditioned room."}],
        "TX": [{"event": "Flood Watch", "areaDesc": "Harris County", "severity": "Moderate",
                "description": "Heavy rain may cause flooding of low-lying areas.",
                "instruction": "Turn around, don't drown."},
               {"event": "Heat Advisory", "areaDesc": "Bexar County", "severity": "Minor",
                "description": "Heat index values up to 108 expected."}],
        "NY": [],
    },
    "points": {  # "lat,lon" rounded to 2 decimals -> forecast periods
        "38.58,-121.49": {"city": "Sacramento", "periods": [
            {"name": f"Period {i}", "temperature": 100 - 3 * i, "temperatureUnit": "F", "windSpeed": "5 mph",
             "windDirection": "SW", "detailedForecast": "Sunny and hot." if i % 2 == 0 else "Clear."}
            for i in range(1, 9)]},
        "40.71,-74.01": {"city": "New York", "periods": [
            {"name": "Tonight", "temperature": 61, "temperatureUnit": "F", "windSpeed": "10 mph",
             "windDirection": "NE", "detailedForecast": "Mostly cloudy."}]},
    },
}
data_path.write_text(json.dumps(nws, indent=1))
os.environ["WEATHER_DATA"] = str(data_path)          # used when we load the server in-process
print("dataset:", data_path, f"({data_path.stat().st_size} bytes)")

### 2. 服务器文件
这就是教程里的 `weather.py`，外加三处升级：一个只加载一次数据集的<strong>生命周期钩子（Lifespan）</strong>、`Annotated` 字段描述与取值范围，以及通过上下文对象（`Context`）报告进度。所有日志都写到 <strong>stderr</strong>。`NOISY` 开关会加入两个故意的 `print()` bug，我们会在第 6 节触发它们。

In [ ]:
%%writefile {server_path}
"""Offline weather MCP server, shaped like the official tutorial."""
import json
import logging
import os
import sys
from contextlib import asynccontextmanager
from dataclasses import dataclass
from typing import Annotated

from pydantic import Field

from mcp.server import MCPServer
from mcp.server.mcpserver import Context

logging.basicConfig(level=logging.INFO, stream=sys.stderr,           # stderr, never stdout
                    format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("weather")

if os.environ.get("NOISY"):
    print("Weather server starting...", flush=True)   # BUG on stdio: not a JSON-RPC message


@dataclass
class WeatherData:
    alerts: dict
    points: dict


@asynccontextmanager
async def lifespan(server: MCPServer):
    path = os.environ["WEATHER_DATA"]       # absolute path, set by the host config's "env"
    logger.info("lifespan: loading %s", path)
    with open(path) as f:
        raw = json.load(f)
    yield WeatherData(alerts=raw["alerts"], points=raw["points"])
    logger.info("lifespan: shutting down")


mcp = MCPServer("weather", lifespan=lifespan)


def format_alert(a: dict) -> str:
    return (f"Event: {a.get('event', 'Unknown')}\nArea: {a.get('areaDesc', 'Unknown')}\n"
            f"Severity: {a.get('severity', 'Unknown')}\n"
            f"Description: {a.get('description', 'No description available')}\n"
            f"Instructions: {a.get('instruction', 'No specific instructions provided')}")


@mcp.tool()
async def get_alerts(state: Annotated[str, Field(description="Two-letter US state code, e.g. CA or NY")],
                     ctx: Context) -> str:
    """Get active weather alerts for a US state."""
    if os.environ.get("NOISY"):
        print(f"looking up alerts for {state}")    # BUG: where does this end up? (section 6)
    data: WeatherData = ctx.request_context.lifespan_context
    logger.info("get_alerts state=%s", state)
    features = data.alerts.get(state.upper())
    if features is None:
        return f"Unable to fetch alerts for {state!r}. Use a two-letter US state code such as CA."
    if not features:
        return "No active alerts for this state."
    return "\n---\n".join(format_alert(a) for a in features)


@mcp.tool()
async def get_forecast(latitude: Annotated[float, Field(description="Latitude of the location", ge=-90, le=90)],
                       longitude: Annotated[float, Field(description="Longitude of the location", ge=-180, le=180)],
                       ctx: Context) -> str:
    """Get the weather forecast (next 5 periods) for a US location."""
    data: WeatherData = ctx.request_context.lifespan_context
    await ctx.report_progress(0, 2, "resolving grid point")
    grid = data.points.get(f"{latitude:.2f},{longitude:.2f}")
    if grid is None:
        return "Unable to fetch forecast data for this location (only US locations are supported)."
    await ctx.report_progress(1, 2, "fetching forecast periods")
    periods = [f"{p['name']}:\nTemperature: {p['temperature']}°{p['temperatureUnit']}\n"
               f"Wind: {p['windSpeed']} {p['windDirection']}\nForecast: {p['detailedForecast']}"
               for p in grid["periods"][:5]]                                # only the next 5 periods
    await ctx.report_progress(2, 2, "done")
    return "\n---\n".join(periods)


if __name__ == "__main__":
    if len(sys.argv) == 3 and sys.argv[1] == "http":
        mcp.run(transport="streamable-http", host="127.0.0.1", port=int(sys.argv[2]))
    else:
        mcp.run(transport="stdio")

### 3. SDK 根据你的类型注解生成了什么
把文件作为模块加载，并在进程内连接。看看 `tools/list` 的线路形式（Wire Form）。文档字符串变成了 `description`。`Annotated` 元数据变成了属性的 `description`、`minimum` 和 `maximum`。`ctx` 参数不见了。`-> str` 返回值生成了一个带 `result` 字段的 `outputSchema`。

In [ ]:
import importlib.util
import logging

from mcp import Client

spec = importlib.util.spec_from_file_location("weather_server", server_path)
weather = importlib.util.module_from_spec(spec)
spec.loader.exec_module(weather)
logging.getLogger("weather").setLevel(logging.WARNING)       # keep the notebook output tidy

async with Client(weather.mcp) as client:
    listed = await client.list_tools()
    wire = listed.model_dump(by_alias=True, exclude_none=True, mode="json")
    print(json.dumps(wire["tools"][1], indent=2))
    print("\nall tools:", [t.name for t in listed.tools])

### 4. 调用工具，并接收进度
`get_forecast` 调用了 `ctx.report_progress`。只有客户端主动请求，才会收到进度通知（Progress Notification）：传入 `progress_callback` 会让 Python 客户端为这一个请求附上 `progressToken`。

In [ ]:
progress_seen = []


async def on_progress(progress, total, message):
    progress_seen.append(f"{progress}/{total} {message}")


async with Client(weather.mcp) as client:
    r = await client.call_tool("get_forecast", {"latitude": 38.58, "longitude": -121.49},
                               progress_callback=on_progress)
    print(r.content[0].text.split("\n---\n")[0], "\n...")
    print("periods returned:", r.content[0].text.count("Temperature"), "(capped at 5 of 8)")
    print("progress notifications:", progress_seen)

    r = await client.call_tool("get_alerts", {"state": "TX"})
    print("\nTX alerts:\n" + r.content[0].text)

    bad = await client.call_tool("get_forecast", {"latitude": 123, "longitude": 0})
    print("\nlatitude=123 -> isError:", bad.is_error, "|", bad.content[0].text.splitlines()[1:3])

最后一次调用根本没有到达你的函数。Pydantic 强制执行了 `le=90`，SDK 把这次失败转换成了一个模型可以读懂并据此纠正的 `isError: true` 结果。

### 5. 真正通过 stdio 运行，并生成宿主风格的日志文件
像 Claude Desktop 这样的宿主（Host）会<strong>拉起</strong>服务器，并把它的 stderr 写入 `mcp-server-weather.log`。我们也这样做：`stdio_client(params, errlog=file)` 是一个可以交给 `Client` 的传输（Transport）。注意其中的 `env`（对应宿主配置的 `"env"` 键）以及脚本的绝对路径。

In [ ]:
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

params = StdioServerParameters(command=sys.executable, args=[str(server_path)],
                               env={"WEATHER_DATA": str(data_path)})
log_file = workdir / "mcp-server-weather.log"

with open(log_file, "w") as errlog:
    async with Client(stdio_client(params, errlog=errlog)) as client:
        print("connected over stdio, protocol", client.protocol_version, "server", client.server_info.name)
        for state in ["CA", "NY"]:
            r = await client.call_tool("get_alerts", {"state": state})
            print(f"{state}:", r.content[0].text.splitlines()[0])

print("\n--- mcp-server-weather.log (the server's stderr) ---")
print(log_file.read_text())

日志显示，生命周期钩子在工具调用之前<strong>只加载了一次</strong>数据，并在客户端关闭 stdin 时完成关闭。

### 6. 弄坏它：stdout 上一句多余的 `print()`
现在设置 `NOISY=1`，手动与服务器对话。服务器写到 stdout 的第一样东西不是 JSON。一个对每一行都执行 `json.loads(line)` 的严格客户端会立刻失败。

In [ ]:
import subprocess

META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "strict-client", "version": "0"}}

proc = subprocess.Popen([sys.executable, str(server_path)], text=True,
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                        env={**os.environ, "NOISY": "1", "WEATHER_DATA": str(data_path)})
proc.stdin.write(json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tools/call",
                             "params": {"name": "get_alerts", "arguments": {"state": "NY"}, "_meta": META}}) + "\n")
proc.stdin.flush()

for n in range(1, 4):                       # read until the response with id 1 arrives
    line = proc.stdout.readline()
    try:
        msg = json.loads(line)
    except json.JSONDecodeError as exc:
        print(f"line {n}: NOT JSON -> {line.strip()!r}  ({exc.msg}) <- a strict client dies here")
        continue
    print(f"line {n}: valid JSON-RPC, id={msg.get('id')} -> {msg['result']['content'][0]['text']!r}")
    if msg.get("id") == 1:
        break

proc.stdin.close()                          # the client hangs up -> the server shuts down
proc.wait(timeout=10)
late_stdout, stderr_text = proc.stdout.read(), proc.stderr.read()
print("\nwritten to stdout AFTER the last response:", repr(late_stdout))
print("in-tool print found in stderr?", "looking up alerts for NY" in stderr_text)

仔细阅读这两个结果：

- <strong>启动时</strong>的 `print()` 在 `mcp.run()` 之前执行，所以它污染了协议通道。
- <strong>工具内</strong>的 `print()`（未刷新）停留在 Python 的 stdout 缓冲区里，并在关闭时被写入了<em>协议流</em>。SDK 2.x 在提供服务期间让 fd 1 指向 stderr，但退出时会恢复真正的 stdout，而缓冲区是在那之后才刷新的。给那个 print 加上 `flush=True` 再运行一次：这次它会落到 stderr 中。无论哪种情况，结果都取决于缓冲行为，这正是你在 stdio 上永远不该 print 的原因。
- 启动时的那个 bug 使用了 `flush=True`，以便它能确定性地出现。如果没有它，当 stdout 是管道时，Python 会对 stdout 进行块缓冲，这段多余的文本会在之后某个无法预测的时刻出现，甚至可能出现在一条消息的中间。那就更难调试了。

Python SDK 自带的客户端比较宽容（它会记录一条解析错误并跳过这一行）。其他宿主没有义务这样做。规则不变：在 stdio 上，日志写到 stderr。

### 7. 同一个服务器跑在 Streamable HTTP 上
无需修改代码：用 `http PORT` 参数运行文件，然后连接到 `http://127.0.0.1:PORT/mcp`。我们故意保留了 `NOISY=1`。在 HTTP 上，stdout 只是一个控制台，所以这些 print 是无害的。

In [ ]:
import socket
import time

for noisy in ("httpx", "httpx2"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]

http_log = open(workdir / "http-server.log", "w")
http_proc = subprocess.Popen([sys.executable, str(server_path), "http", str(port)],
                             env={**os.environ, "NOISY": "1", "WEATHER_DATA": str(data_path)},
                             stdout=http_log, stderr=subprocess.STDOUT)
for _ in range(100):                                     # wait until uvicorn accepts connections
    try:
        socket.create_connection(("127.0.0.1", port), timeout=0.2).close()
        break
    except OSError:
        time.sleep(0.1)

try:
    url = f"http://127.0.0.1:{port}/mcp"
    for attempt in range(2):                             # two independent clients, one server process
        async with Client(url) as client:
            r = await client.call_tool("get_alerts", {"state": "CA"})
            print(f"client {attempt + 1}: {client.protocol_version} ->", r.content[0].text.splitlines()[0])
finally:
    http_proc.terminate()
    http_proc.wait(timeout=10)
    http_log.close()

print("\n--- server console (stdout + stderr) ---")
print("\n".join(l for l in (workdir / "http-server.log").read_text().splitlines() if "POST" not in l))

控制台中能看到那句 noisy 的 `print`、uvicorn 自己的消息，以及整个进程只发生了一次的生命周期加载，它被两个客户端共享。

### 8. 生成 Claude Desktop 配置
一个小助手函数按照教程推荐的方式（`uv --directory <abs> run weather.py`）写出 `mcpServers` 条目。它还会写出直接调用解释器的变体，并对结果进行<strong>检查（lint）</strong>，找出典型错误：相对路径、GUI 应用可能找不到的裸命令，以及缺失的 `env`。

In [ ]:
import shutil


def desktop_entry(server_file: Path, use_uv: bool = True) -> dict:
    env = {"WEATHER_DATA": str(data_path)}
    if use_uv:
        return {"command": shutil.which("uv") or "uv",
                "args": ["--directory", str(server_file.parent), "run", server_file.name], "env": env}
    return {"command": sys.executable, "args": [str(server_file)], "env": env}


def lint_entry(name: str, entry: dict) -> list[str]:
    problems = []
    if not os.path.isabs(entry["command"]):
        problems.append(f"{name}: command {entry['command']!r} is not absolute; GUI hosts may not see your PATH")
    for arg in entry.get("args", []):
        if ("/" in arg or "\\" in arg) and not os.path.isabs(arg):
            problems.append(f"{name}: relative path {arg!r}; the host's working directory may be '/'")
    for key, value in entry.get("env", {}).items():
        if value.endswith(".json") and not os.path.isabs(value):
            problems.append(f"{name}: env {key} is a relative path")
    return problems


config = {"mcpServers": {"weather": desktop_entry(server_path, use_uv=True),
                         "weather-python": desktop_entry(server_path, use_uv=False),
                         "broken": {"command": "uv", "args": ["--directory", "./weather", "run", "weather.py"]}}}
print(json.dumps(config, indent=2))
print("\nlint:")
for name, entry in config["mcpServers"].items():
    print(" ", name, "->", lint_entry(name, entry) or "ok")
print("\nmacOS file: ~/Library/Application Support/Claude/claude_desktop_config.json  (then fully quit Claude)")

### 9. 底层原理：六个步骤，配一个脚本化的“Claude”
把教程中“what's happening under the hood”的列表写成代码。一个确定性的模型替身（脚本化规划器（Scripted Planner））根据所公布的描述选择工具；宿主通过 MCP 执行它，并把结果回传。如果设置了 `ANTHROPIC_API_KEY`，第 2 步和第 5 步将由真实模型完成。

In [ ]:
async def fake_model(question: str, tools) -> dict:
    """Step 2: choose a tool by reading names/descriptions (a stand-in for the LLM)."""
    q = question.lower()
    if "alert" in q:
        state = [w.strip("?.") for w in question.split() if w.strip("?.").isupper() and len(w.strip("?.")) == 2][0]
        return {"name": "get_alerts", "arguments": {"state": state}}
    return {"name": "get_forecast", "arguments": {"latitude": 38.58, "longitude": -121.49}}


question = "Are there any weather alerts in TX?"
async with Client(weather.mcp) as client:
    tools = (await client.list_tools()).tools
    print("1. host sends the question + tool list to the model:", question)
    choice = await fake_model(question, tools)
    print("2. model chooses:", choice)
    result = await client.call_tool(choice["name"], choice["arguments"])
    print("3. host executes it through MCP -> isError:", result.is_error)
    print("4. result goes back to the model:", result.content[0].text.count("Event:"), "alerts")
    events = [l.split(": ", 1)[1] for l in result.content[0].text.splitlines() if l.startswith("Event:")]
    print("5./6. answer shown to the user: Texas has", len(events), "active alerts:", ", ".join(events))

if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    llm = anthropic.Anthropic()
    async with Client(weather.mcp) as client:
        tool_defs = [{"name": t.name, "description": t.description, "input_schema": t.input_schema}
                     for t in (await client.list_tools()).tools]
        messages = [{"role": "user", "content": "What's the weather in Sacramento (38.58, -121.49)?"}]
        response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=tool_defs, messages=messages)
        while response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type == "tool_use":
                    r = await client.call_tool(block.name, block.input)
                    results.append({"type": "tool_result", "tool_use_id": block.id,
                                    "content": r.content[0].text, "is_error": bool(r.is_error)})
            messages.append({"role": "user", "content": results})
            response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=tool_defs, messages=messages)
        print("\nClaude:", "".join(b.text for b in response.content if b.type == "text"))
else:
    print("\n(skipped real model: set ANTHROPIC_API_KEY to let Claude do steps 2 and 5)")

### 10. 进度之外的 Context 对象
一个小型配套服务器展示了上下文对象（`Context`）的其余功能。它在工具内部读取自己的一个资源，检查请求元数据，并使用<strong>已弃用的</strong>协议日志（Protocol Logging）。协议日志只会送达通过 `log_level` 选择启用的 2026-07-28 客户端。

In [ ]:
from mcp.server import MCPServer
from mcp.server.mcpserver import Context

ctx_demo = MCPServer("ctx-demo")


@ctx_demo.resource("config://units")
def units() -> str:
    return "imperial"


@ctx_demo.tool()
async def describe_request(ctx: Context) -> dict[str, str]:
    """Report what the Context object knows about this request."""
    units_text = list(await ctx.read_resource("config://units"))[0].content
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        await ctx.info("describe_request called")          # deprecated notifications/message
    return {"request_id": ctx.request_id, "protocol": ctx.protocol_version or "", "units": units_text}


for level in [None, "info"]:
    received = []

    async def on_log(params):
        received.append(f"[{params.level}] {params.data}")

    async with Client(ctx_demo, logging_callback=on_log, log_level=level) as client:
        r = await client.call_tool("describe_request", {})
        print(f"log_level={level!r:7} -> structured={r.structured_content} | log messages received={received}")

### 动手练习
1. 在服务器文件中添加第三个工具 `get_station_count(state: Literal["CA", "TX", "NY"]) -> int`。重新运行第 2–3 节，在它的 `inputSchema` 中找到 `enum`，并找到整数类型的 `outputSchema`。
2. 把 `NOISY` 启动时的 `print` 改为写到 `sys.stderr`（`print(..., file=sys.stderr)`），然后重新运行第 6 节。现在 stdout 上的第一行是什么？
3. 让生命周期钩子失败（在第 5 节的 `env` 中把 `WEATHER_DATA` 指向一个不存在的文件）。客户端抛出了什么异常？真正的原因记录在 `mcp-server-weather.log` 的哪里？
4. 修改 `desktop_entry`，让它同时输出 Windows 形式的路径（双反斜杠），并扩展 `lint_entry`，使其能发现缺失 `WEATHER_DATA` 的 `env` 条目。